# L28 · Comunicar un resultado causal

**S28 · Lectura: Cap. 32 · 40–55 min**

## La pregunta

¿Cómo explicamos el efecto sin prometer más de lo que sabemos?

## Qué harás

Escribir un mensaje con población, intervalo, supuesto y límite.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Un resumen de una página para quien decide.

> **Idea clave:** La forma de comunicar puede borrar los supuestos; por eso deben aparecer desde la primera frase.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación: el artefacto A2 (y nada de números por defecto)

In [ ]:
from lab_common import *
import numpy as np, pandas as pd, re
rng = setup()
art = a2_fit()          # lee artefactos/a2.json si existe; si no, recalcula desde los datos. Nunca devuelve un número inventado
th, (lo, hi), se = art["theta"], art["ic95"], art["se"]
es = lambda x, d=2: f"{x:.{d}f}".replace(".", ",")
print(f"A2 → ATE = {es(th)} (EE {es(se)}), IC 95 % = [{es(lo)}; {es(hi)}] · método: {art['metodo']} · datos: {art['origen_datos']} · n = {art['n_train']}")

> **Regla de este notebook.** Una versión anterior devolvía `3,53 [2,89; 4,17]` «por defecto» si el artefacto faltaba. Eso es peor que fallar: un número plausible pero inventado pasaría a un resumen ejecutivo. Aquí, si el artefacto no se puede leer ni recalcular, el notebook **falla**.

## 1. La misma estimación, tres redacciones
Tres versiones del mismo resultado. Antes de ver nada más, léelas como lo haría quien decide: ¿qué haría con cada una?

In [ ]:
POB = "los niños del escenario de práctica (IHDP, sintético)"
V = {
 "A. solo el número": f"El programa aumenta el puntaje en {es(th, 1)} puntos.",
 "B. número, intervalo y supuestos": f"El programa aumenta el puntaje en {es(th)} puntos (IC 95 %: {es(lo)}–{es(hi)}), estimado con DML sobre la población que efectivamente entró en los datos, bajo el supuesto de que no hay factores no observados que determinen a la vez quién recibió el programa y su puntaje.",
 "C. tres capas": f"Entre {POB}, el programa subió el puntaje unos {es(th, 1)} puntos (entre {es(lo, 1)} y {es(hi, 1)}).\n\nEl resultado supone que no hay factores ocultos que influyan a la vez en quién recibió el programa y en su puntaje. No implica que el efecto sea el mismo en niños fuera del rango estudiado.\n\nDetalle técnico: ver la model card y el notebook A2."}
for k, t in V.items(): print(f"[{k}]\n{t}\n")

**Mira el resultado.** ¿Qué hace cada versión con la atención de quien lee, y cuál de las tres sobrevive mejor a «varias capas de resumen»?

**Respuesta:**

**A** es la que sobrevive a los resúmenes (es la única que cabe en una diapositiva), y por eso es la que termina circulando: dice «aumenta» sin población, sin intervalo y sin condición. **B** es honesta pero tiene el riesgo que el capítulo nombra: sin cuidado en la forma suena a «el equipo no está seguro de nada», y quien resume la recorta a **A**. **C** estructura el mensaje: lo que se recuerda (capa 1) ya incluye la población y un rango; el supuesto y lo que **no** implica están en un lugar visible y corto (capa 2); el detalle está enlazado (capa 3). No se elige un extremo: se **ordena** la información.



## 2. Frecuencias naturales y riesgo absoluto frente a relativo (Spiegelhalter)
«El efecto es de 3,6 puntos (IC…)» se retiene peor que «de cada 100 niños…». Para pasar un efecto medio a frecuencias hace falta un resultado **binario**: definimos «alcanza el umbral $u$» y aplicamos el efecto del artefacto.

> **Supuesto explícito de este cálculo:** el efecto es **aditivo y común** (a cada niño se le suma $\hat\theta$). Es una simplificación (el efecto real varía entre niños, L13–L14): se usa aquí para ilustrar la comunicación, y habría que decirlo en el texto.

In [ ]:
sc = load_scenario("synthetic/inference/ihdp_medium")
y0 = sc.train.loc[sc.train[sc.treatment] == 0, sc.outcome].to_numpy(float)
def frec(q_base, efecto):
    u = np.quantile(y0, 1 - q_base)                                  # umbral: lo supera la fracción q_base de los controles
    p0 = float((y0 > u).mean()); p1 = float((y0 + efecto > u).mean()); return p0, p1
filas = {}
for etq, ef, e_lo, e_hi in (("efecto de A2", th, lo, hi), ("HIPOTÉTICO: efecto 10 veces menor", th / 10, lo / 10, hi / 10)):
    for q in (0.50, 0.05, 0.01):
        p0, p1 = frec(q, ef); a_lo, a_hi = frec(q, e_lo)[1], frec(q, e_hi)[1]
        filas[f"{etq} · base {100 * q:.0f} %"] = {"sin programa (de cada 100)": f"{100 * p0:.1f}", "con programa (de cada 100)": f"{100 * p1:.1f}", "diferencia absoluta (de cada 100)": f"{100 * (p1 - p0):.1f}",
                                                  "entre (IC)": f"{100 * (a_lo - p0):.1f} y {100 * (a_hi - p0):.1f}", "riesgo relativo": f"{p1 / p0:.1f}"}
show(pd.DataFrame(filas).T)

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

**Mira el resultado.** Mira primero el efecto hipotético diez veces menor. ¿Qué pasa con el riesgo relativo cuando la base es 1 % y cuál cifra comunicarías?

**Respuesta:**

Con un efecto pequeño y una base baja, el **relativo** se ve grande (en la fila de base 1 %, más del doble) mientras que lo que cambió en la vida real es **poco más de un niño de cada 100**: «más del doble de probabilidad» es correcto y engañoso. Se comunica el **absoluto** y en frecuencias naturales («de cada 100 niños, N más alcanzan el umbral»), siempre junto al intervalo en las mismas unidades (columna «entre»), que dice la incertidumbre sin pronunciar «intervalo de confianza». Con el efecto real de A2, que es grande en relación con la dispersión de Y, el absoluto también es grande y los dos relatos coinciden en lo importante; la regla de dar ambos es independiente de eso: es la que te protege cuando el efecto es chico.



## 3. Completa tú: la plantilla de tres capas desde el artefacto
Completa `plantilla_tres_capas(art, poblacion, supuesto_clave, no_implica, unidad="puntos")` para que devuelva el texto del Listado 32.1 con **todas las cifras tomadas de `art`** (nada escrito a mano, nada por defecto):

* `## Hallazgo`: una oración, sin jerga, con la población y el efecto.
* `## Lo que esto significa (y lo que no)`: población, principal supuesto, incertidumbre (IC 95 %) y «Este resultado NO implica».
* `## Detalle técnico`: enlace o referencia al model card/notebook, método y origen de los datos.

In [ ]:
# ✏️ Completa tú — plantilla_tres_capas
# Pista: Un f-string multilínea. Hallazgo: una oración con la población y th_ (y el rango). Incertidumbre: es(lo_) y es(hi_). Detalle: método, origen de los datos y n del artefacto.
def plantilla_tres_capas(art, poblacion, supuesto_clave, no_implica, unidad="puntos", enlace="model_card.md"):
    th_, (lo_, hi_) = art["theta"], art["ic95"]          # las cifras salen de aquí, de ningún otro lado
    # Devuelve un str con las tres secciones del Listado 32.1: "## Hallazgo", "## Lo que esto significa (y lo que no)", "## Detalle técnico".
    # Usa es(x, d) para formatear con coma decimal.
    raise NotImplementedError("completa plantilla_tres_capas")

In [ ]:
SUP = "no hay factores ocultos que influyan a la vez en quién recibió el programa y en su puntaje"
NOIMP = "que el efecto sea el mismo para niños fuera del rango estudiado, ni que cada niño gane lo mismo"
txt = plantilla_tres_capas(art, POB, SUP, NOIMP)
norm = lambda t: t.replace(",", ".")
def cifras_ok(t): return all(f"{v:.2f}" in norm(t) or f"{v:.1f}" in norm(t) for v in (th, lo, hi))
lab_check("plantilla_tres_capas", lambda: ("## Hallazgo" in txt and "## Lo que esto significa" in txt and "## Detalle" in txt, cifras_ok(txt),
                                          ("3.53" not in norm(txt)) or abs(th - 3.53) < 0.005, SUP in txt and NOIMP in txt, len(txt.split("## Hallazgo")[1].split("##")[0].strip().split("\n")) == 1),
          [((), (True,) * 5, 0)])
print(txt)

**Mira el resultado.** ¿Por qué la capa 1 lleva la población y un rango, y no solo «aumentó en 3,6»?

**Respuesta:**

Porque la capa 1 es **lo que se recuerda y lo que se cita**: Spiegelhalter lo dice como regla (la audiencia asume por defecto que el número aplica a *su* población). Si la población y el rango no están en la oración que sobrevive, desaparecen. La capa 2 sigue siendo necesaria porque ahí cabe el *supuesto* y el «NO implica», que no caben en una oración sin volverla ilegible.



## 4. Checklist de las 4 señales sobre un resumen ejecutivo
El §32.6 da cuatro señales. Las convertimos en una **heurística de palabras clave** (¡es una ayuda, no un juez!) y la aplicamos a un resumen ejecutivo ficticio con los defectos típicos.

In [ ]:
def checklist_comunicacion(texto):
    t = texto.lower(); primera = re.split(r"(?<=[.!?])\s|\n", texto.strip())[0]
    tiene_sup = bool(re.search(r"supuest|supone|suponiendo|asumiendo|siempre que", t)); tiene_ic = bool(re.search(r"\bic\b|intervalo|entre .{1,12} y |±|\[[\d,;. -]+\]", t))
    pobl = bool(re.search(r"poblaci|válido para|aplica a|solo para|en el grupo|entre los|entre las|niños|hogares|participantes|pacientes", t))
    causal = bool(re.search(r"\bcausa\b|provoca|produce|gracias al|demuestra que|aumenta|reduce|mejora|subió|redujo", t))
    jerga = bool(re.search(r"dml|cross-fit|ortogonal|\bse\b|\bate\b|propensity|nuisance|hc1|\bic\b", primera.lower()))
    return {"1. no menciona supuesto ni intervalo": not (tiene_sup or tiene_ic),
            "2. no dice para qué población vale": not pobl,
            "3. lenguaje causal sin ninguna condición": causal and not tiene_sup,
            "4. no hay versión accesible (la primera frase tiene jerga o no hay capas)": jerga or ("##" not in texto and "\n\n" not in texto)}
RESUMEN = "Resumen ejecutivo. El modelo DML (cross-fitting K=5, ExtraTrees) estima un ATE de 3,5 con SE = 0,31 bajo ortogonalidad de Neyman: el programa causa un aumento del puntaje y recomendamos escalarlo a todas las regiones."
rows = {"resumen ejecutivo ficticio": checklist_comunicacion(RESUMEN), "versión A (solo el número)": checklist_comunicacion(V["A. solo el número"]), "versión B (número, intervalo y supuestos)": checklist_comunicacion(V["B. número, intervalo y supuestos"]),
        "versión C (tres capas)": checklist_comunicacion(V["C. tres capas"]), "plantilla completada": checklist_comunicacion(txt)}
show(pd.DataFrame(rows).T.replace({True: "⚠ sí", False: "—"}))

**Mira el resultado.** ¿Qué señales enciende el resumen ficticio, y por qué la versión B también enciende alguna?

**Respuesta:**

El resumen ficticio enciende las cuatro: sin supuesto ni intervalo legibles, sin población, con «causa» sin condición y con jerga en la primera frase («DML», «ATE», «SE»). La versión B enciende la **4**: está bien como párrafo técnico, pero no tiene una capa accesible separada. Es el aviso de que **una comunicación honesta puede seguir siendo inutilizable** si no está estructurada. (Y recuerda el límite de la heurística: un texto puede incluir la palabra «supuesto» y no decir cuál; las cuatro señales ayudan a notar lo que falta, no certifican lo que hay.)



## 5. Una asimetría que justifica inclinarse por matizar
Sobreestimar la certeza produce decisiones que fallan **en silencio** («el programa se implementa, no funciona, y rara vez se atribuye el fracaso al análisis»); subestimarla produce, en el peor caso, una decisión pospuesta. Los dos errores tienen costo, pero solo el primero se disfraza de éxito.

In [ ]:
# Ejercicio de lectura (sin cifras inventadas): relee tus tres redacciones y responde por escrito
for q in ("1. ¿Qué decisión tomaría alguien que lee solo la versión A y qué no sabría para tomarla bien?",
          "2. ¿Qué frase de la versión B quitaría quien tiene que resumirla a una línea? ¿Está en la capa 1 de C?",
          "3. ¿Para qué población NO vale el resultado de A2? (pista: el rango de covariables del artefacto, art['soporte'])"): print(q)
print("\nrango de covariables que vio el modelo (soporte):", {k: [round(a, 1), round(b, 1)] for k, (a, b) in list(art["soporte"].items())[:3]}, "…")

## 6. Supuesto y bitácora: el que más pesa, en lenguaje llano
El supuesto de mayor peso para A2 es la **no confusión** (que no haya factores ocultos que determinen a la vez quién recibe el programa y el resultado). Su versión para una persona no técnica es la frase de la capa 2.

In [ ]:
fila_b = bitacora(
    pregunta="¿Qué debe entender quien decide al leer el resultado de A2, y qué se le dice que no debe inferir?",
    metodo="Plantilla de tres capas con cifras tomadas del artefacto; frecuencias naturales y riesgo absoluto junto al relativo; checklist de 4 señales",
    estimando="El ATE del artefacto A2",
    numero_ic=f"ATE {es(th)} puntos, IC 95 % [{es(lo)}; {es(hi)}], extraído de artefactos/a2.json (no escrito a mano)",
    supuesto=supuesto("no observable", "declarado y acotado", "no hay factores ocultos que influyan a la vez en quién recibió el programa y en su puntaje; se dice en la capa 2, junto con la población y lo que el resultado NO implica"),
    decision="continuar", lab="L28")

---
## Qué queda para el libro
Quedan en el libro: la visualización de incertidumbre (Spiegelhalter, 2019; Galesic y Garcia-Retamero, 2011), la discusión de interpretabilidad de Lipton (2018) y el ejercicio de cierre del proyecto final. El checklist de este notebook es una **heurística de palabras clave**: ayuda a notar lo que falta, no reemplaza leer el texto.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*